In [ ]:
import anndata as ad
import scipy.sparse as sp
import numpy as np
import pandas as pd
multiome_data = ad.read_h5ad(
    "../../data/multiome/zf_multiome_atlas_full_ATAC_v1_release.h5ad"
)
multiome_data


AnnData object with n_obs × n_vars = 94562 × 640834
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'global_annotation', 'nCount_peaks_bulk', 'nFeature_peaks_bulk', 'nCount_peaks_celltype', 'nFeature_peaks_celltype', 'nCount_peaks_merged', 'nFeature_peaks_merged', 'SCT.weight', 'peaks_merged.weight', 'nCount_Gene.Activity', 'nFeature_Gene.Activity', 'nCount_peaks_integrated', 'nFeature_peaks_integrated', 'dataset', 'integrated.weight', 'peaks_integrated.weight', 'wsnn_res.0.8', 'seurat_clusters', 'leiden_0.5', 'leiden_0.8', 'leiden_1', 'leiden_1.2', 'leiden_1.5', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'leiden_6', 'leiden_7', 'leiden_8', 'leiden_9', 'leiden_10', 'leiden_0.5_merged', 'leiden_0.8_merged', 'leiden_1_merged', 'leiden_1.2_merged', 'leiden_1.5_merged', 'leiden_2_merged', 'leiden_3_merged', 'leiden_4_merged', 'leiden_5_me

In [9]:
celltype_col = "annotation_ML_coarse"  

adata = multiome_data  
# choose counts matrix
counts = adata.layers["counts"] if "counts" in adata.layers else adata.X

# cell type labels
labels = adata.obs[celltype_col].astype("category")
categories = labels.cat.categories
codes = labels.cat.codes.to_numpy()

# design matrix: cells × celltypes (one-hot)
row = np.arange(adata.n_obs)
col = codes
data = np.ones(adata.n_obs, dtype=np.int8)
design = sp.csr_matrix((data, (row, col)), shape=(adata.n_obs, len(categories)))

# pseudobulk: (celltypes × cells) @ (cells × peaks) → (celltypes × peaks)
pseudobulk = design.T @ counts  # shape: (n_celltypes, n_peaks)


# convert to pandas DataFrame (be careful, may be huge!)
pb_df = pd.DataFrame(
    pseudobulk.toarray().T,     # transpose so peaks = rows
    index=adata.var_names,      # peaks
    columns=categories          # celltypes
)

pb_df.head()

,NMPs,PSM,differentiating_neurons,endocrine_pancreas,endoderm,enteric_neurons,epidermis,fast_muscle,floor_plate,hatching_gland,...,neural_telencephalon,neurons,notochord,optic_cup,pharyngeal_arches,primordial_germ_cells,pronephros,somites,spinal_cord,tail_bud
1-32-526,11,28,14,12,12,4,51,10,20,8,...,4,12,2,29,15,0,11,21,32,13
1-2372-3057,17,52,26,32,22,12,162,21,57,18,...,27,57,9,81,26,1,41,52,55,35
1-3427-4032,55,154,62,77,34,41,376,73,110,45,...,31,135,20,193,86,5,103,115,169,92
1-4469-7268,670,1999,371,893,491,161,2694,904,801,453,...,327,1065,233,1478,654,16,1154,1296,1496,1323
1-9541-9969,152,493,80,179,115,31,541,167,155,64,...,66,196,57,261,169,2,259,241,285,380
